# Evaluate Baseline LSTM on Google Colab

This notebook evaluates an existing baseline LSTM checkpoint using the existing `src/training/lstm_training/evaluate.py` script. It does not train, recreate data, or duplicate evaluation logic.

Select a GPU runtime if available: `Runtime > Change runtime type > GPU`.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Set Project Paths

This assumes the project root is `/content/drive/MyDrive/Seoul_bike_project`. The dataset is copied from Drive to local Colab disk before evaluation.

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path("/content/drive/MyDrive/Seoul_bike_project")
DRIVE_DATA_DIR = PROJECT_DIR / "data" / "lstm_baseline"
LOCAL_DATA_DIR = Path("/content/lstm_baseline")
DATA_DIR = LOCAL_DATA_DIR
CONFIG_PATH = PROJECT_DIR / "configs" / "lstm_baseline.yaml"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints" / "lstm_baseline"
MODEL_DIR = PROJECT_DIR / "models" / "lstm_baseline"
LOG_DIR = PROJECT_DIR / "logs" / "lstm_baseline"
BEST_CHECKPOINT = CHECKPOINT_DIR / "best.pt"

os.chdir(PROJECT_DIR)
print("Project directory:", PROJECT_DIR)
print("Current working directory:", Path.cwd())
print("Drive data directory:", DRIVE_DATA_DIR)
print("Local Colab data directory:", DATA_DIR)

## 3. Install Dependencies

In [ ]:
!pip install -q torch numpy pyyaml tqdm wandb

## 4. Verify GPU

## 5. Copy Dataset to Colab Disk

In [ ]:
from pathlib import Path
import shutil

PROJECT_DIR = Path("/content/drive/MyDrive/Seoul_bike_project")
DRIVE_DATA_DIR = PROJECT_DIR / "data" / "lstm_baseline"
LOCAL_DATA_DIR = Path("/content/lstm_baseline")
DATA_DIR = LOCAL_DATA_DIR
RELOAD_LOCAL_DATA = True

if not DRIVE_DATA_DIR.exists():
    raise FileNotFoundError(f"Drive dataset directory does not exist: {DRIVE_DATA_DIR}")

if LOCAL_DATA_DIR.exists() and RELOAD_LOCAL_DATA:
    shutil.rmtree(LOCAL_DATA_DIR)

if not LOCAL_DATA_DIR.exists():
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
    print(f"Copied dataset from {DRIVE_DATA_DIR} to {LOCAL_DATA_DIR}")
else:
    print(f"Using existing local dataset at {LOCAL_DATA_DIR}")

print("Training/evaluation data directory:", DATA_DIR)

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is not available. Evaluation will run on CPU unless a GPU runtime is selected.")

## 6. Verify Required Files and Best Checkpoint

In [ ]:
required_files = [
    CONFIG_PATH,
    PROJECT_DIR / "src" / "training" / "lstm_training" / "evaluate.py",
    PROJECT_DIR / "src" / "training" / "lstm_training" / "train_lstm.py",
    DATA_DIR / "dynamic_features.npy",
    DATA_DIR / "targets.npy",
    DATA_DIR / "targets_raw.npy",
    DATA_DIR / "static_numeric.npy",
    DATA_DIR / "splits.json",
    DATA_DIR / "scalers.json",
    DATA_DIR / "feature_config.json",
    DATA_DIR / "dataset_summary.json",
    BEST_CHECKPOINT,
]

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

LOG_DIR.mkdir(parents=True, exist_ok=True)
print("All required files found.")
print("Best checkpoint:", BEST_CHECKPOINT)

## 7. Run Test Evaluation on Best Checkpoint

This uses raw count-scale metrics after inverse-transforming model predictions.

In [ ]:
!python src/training/lstm_training/evaluate.py \
  --config configs/lstm_baseline.yaml \
  --data_dir /content/lstm_baseline \
  --checkpoint_path /content/drive/MyDrive/Seoul_bike_project/checkpoints/lstm_baseline/best.pt \
  --log_dir /content/drive/MyDrive/Seoul_bike_project/logs/lstm_baseline \
  --split test

## 8. Display Test Metrics

In [ ]:
import json

test_metrics_path = LOG_DIR / "test_metrics.json"
final_metrics_path = LOG_DIR / "final_metrics.json"

if test_metrics_path.exists():
    with open(test_metrics_path, "r", encoding="utf-8") as f:
        metrics = json.load(f)
    metrics
elif final_metrics_path.exists():
    with open(final_metrics_path, "r", encoding="utf-8") as f:
        metrics = json.load(f)
    metrics
else:
    print("No test_metrics.json or final_metrics.json found yet.")